# Telco Customer Churn — Preprocessing

Du CSV brut à un jeu de données propre, prêt à modéliser.

In [1]:
%pip install pandas

import pandas as pd

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Phase 0 — Récupérer la donnée et l'ouvrir

In [2]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Forme :", df.shape)
print(df.dtypes)
df.head()

Forme : (7043, 21)
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
## Phase 1 — L'audit qualité

In [3]:
def audit_qualite(df):
    """Affiche un rapport de santé du dataset.

    Doit montrer : forme, types, % de manquants par colonne (triés),
    et la répartition de la cible Churn (en valeur ET en pourcentage).
    """
    print(f"Forme : {df.shape}")
    print()

    manquants = (df.isna().mean() * 100).sort_values(ascending=False)
    cols_avec_trous = manquants[manquants > 0]

    if len(cols_avec_trous) == 0:
        print("Manquants détectés : 0 colonne  (méfiance : des trous sont peut-être cachés, voir Phase 2)")
    else:
        print("Manquants par colonne (%) :")
        for col, pct in cols_avec_trous.items():
            print(f"  {col} : {pct:.1f}%")

    print()

    if "Churn" not in df.columns:
        print("Colonne Churn absente.")
        return

    if len(df) == 0:
        print("Dataset vide.")
        return

    repartition = df["Churn"].value_counts()
    total = len(df)

    for val, count in repartition.items():
        pct = 100 * count / total
        print(f"Churn  {val} : {count} ({pct:.1f}%)")

In [4]:
print("=== Cas normal : dataset complet ===")
audit_qualite(df)

=== Cas normal : dataset complet ===
Forme : (7043, 21)

Manquants détectés : 0 colonne  (méfiance : des trous sont peut-être cachés, voir Phase 2)

Churn  No : 5174 (73.5%)
Churn  Yes : 1869 (26.5%)


In [5]:
print("=== Cas limite : une seule classe (Churn == No) ===")
df_une_classe = df[df["Churn"] == "No"]
audit_qualite(df_une_classe)

=== Cas limite : une seule classe (Churn == No) ===
Forme : (5174, 21)

Manquants détectés : 0 colonne  (méfiance : des trous sont peut-être cachés, voir Phase 2)

Churn  No : 5174 (100.0%)


In [6]:
print("=== Cas adversarial : déséquilibre visible ? ===")
repartition = df["Churn"].value_counts(normalize=True) * 100
print(repartition.round(1))
print()
print("→ 73/27 : le churn est minoritaire. L'accuracy seule sera trompeuse demain.")

=== Cas adversarial : déséquilibre visible ? ===
Churn
No     73.5
Yes    26.5
Name: proportion, dtype: float64

→ 73/27 : le churn est minoritaire. L'accuracy seule sera trompeuse demain.


In [ ]:
## Phase 2 — La colonne piégée (TotalCharges)

In [ ]:
def reparer_total_charges(df):
    """Convertit TotalCharges en numérique et traite les trous révélés.

    Doit renvoyer le df réparé et afficher combien de trous ont été démasqués.
    """
    df = df.copy()

    avant = df["TotalCharges"].isna().sum()
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    apres = df["TotalCharges"].isna().sum()
    trous_demasques = apres - avant

    print(f"Trous démasqués : {trous_demasques}")
    print(f"Type après conversion : {df['TotalCharges'].dtype}")

    if trous_demasques > 0:
        mediane = df["TotalCharges"].median()
        df["TotalCharges"] = df["TotalCharges"].fillna(mediane)
        print(f"Imputation par la médiane : {mediane:.2f}")

    return df

In [ ]:
df = reparer_total_charges(df)
print(f"\nForme finale : {df.shape}")
print(df["TotalCharges"].dtype)

In [ ]:
print("=== Edge case : colonne 100% texte ===")
df_test = pd.DataFrame({"col_texte": ["abc", "def", "ghi"]})
converti = pd.to_numeric(df_test["col_texte"], errors="coerce")
if converti.isna().all():
    print("→ 100% non numérique : on refuse de l'utiliser comme feature numérique.")

In [ ]:
print("=== Adversarial : virgule au lieu du point (29,90) ===")
df_test2 = pd.DataFrame({"MonthlyCharges": ["29.90", "29,90", "45.00"]})
converti2 = pd.to_numeric(df_test2["MonthlyCharges"], errors="coerce")
print(converti2)


**Choix : imputation par médiane** (pas suppression des 11 lignes).

- On ne perd aucun client churn.
- 11 lignes sur 7043, impact négligeable.
- La médiane est robuste aux valeurs extrêmes.

## Phase 3 — Encoder les catégorielles

In [ ]:
def encoder_features(df):
    """Encode toutes les colonnes catégorielles.

    Doit renvoyer un df 100% numérique, prêt pour un modèle.
    """
    df = df.copy()
    nb_avant = df.shape[1]

    # customerID = identifiant, pas une feature
    df = df.drop(columns=["customerID"])

    # cible binaire
    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

    # colonnes Yes/No simples → 0/1
    for col in df.columns:
        if df[col].dtype == "object" or str(df[col].dtype) == "str":
            valeurs = set(df[col].dropna().unique())
            if valeurs <= {"Yes", "No"}:
                df[col] = df[col].map({"Yes": 1, "No": 0})

    # le reste (nominales) → One-Hot
    cols_texte = [c for c in df.columns if df[c].dtype == "object" or str(df[c].dtype) == "str"]
    df = pd.get_dummies(df, columns=cols_texte, dtype=int)

    print(f"Colonnes avant : {nb_avant} → après : {df.shape[1]}")
    print(f"Toutes numériques : {df.select_dtypes(exclude='number').shape[1] == 0}")

    return df

In [ ]:
print(f"Avant encodage : {df.shape}")
df = encoder_features(df)
print(f"Après encodage : {df.shape}")
df.head()

**Contract : nominal ou ordinal ?**

Les 3 modalités (Month-to-month, One year, Two year) ont un ordre métier, mais on choisit **One-Hot** : pas de supposer que "Two year" = 2× "One year" en distance. Chaque contrat est une catégorie distincte.

In [ ]:
print("=== Adversarial : si on encodait customerID en One-Hot ===")
print(f"→ {df.shape[0]} colonnes créées (1 par client). Explosion de dimensions.")
print("→ C'est pourquoi on supprime les identifiants avant l'encodage.")